In [ ]:
#
# Project:
#      PyTorch Dojo (https://github.com/wo3kie/ml-dojo)
#
# Author:
#      Lukasz Czerwinski (https://www.lukaszczerwinski.pl/)
#

$$ \text{onehot}(y) = \textbf{Y} = (y_1, y_2, \dots, y_C) $$
$$ y_k = \begin{cases}
y_k = 1 & \text{if } y = k \\
y_k = 0 & \text{otherwise}
\end{cases} $$
$$ \sum_k y_k = 1 $$
$$ y = 0 \implies (1, 0, 0, \dots) $$
$$ y = 1 \implies (0, 1, 0, \dots) $$
$$ y = 2 \implies (0, 0, 1, \dots) $$
$$ \vdots $$
$$ y = C \implies (0, 0, \dots, 1) $$
$$ \diamond \diamond \diamond $$

$$ Z = WX + b $$
$$ \frac{\partial Z}{\partial W} = X $$
$$ \frac{\partial Z}{\partial b} = 1 $$
$$ \diamond \diamond \diamond $$

$$ P = \{p_c : c \in \mathcal{C} \} $$
$$ p_i = softmax(z_i) = \frac{e^{z_i}}{\sum_{k} e^{z_k}} $$
$$
\frac{\partial p_i}{\partial z_j} = p_i (\delta_{ij} - p_j) =
\begin{dcases}
i = j & \frac{\partial p_i}{\partial z_i} = 
\frac{e^{z_i} \sum_{k} e^{z_k} - e^{z_i} e^{z_i}}{\sum_{k} e^{z_k} \sum_{k} e^{z_k}} =
p_i (1 - p_i) \\
\\
i \neq j & \frac{\partial p_i}{\partial z_j} = 
\frac{0 \cdot \sum_{k} e^{z_k} - e^{z_i} e^{z_j}}{\sum_{k} e^{z_k} \sum_{k} e^{z_k}} = 
-p_i p_j  = p_i (0 - p_j)\\
\end{dcases}
$$
$$ \frac{\partial P}{\partial Z} = \begin{pmatrix}
p_0 (1 - p_0) & -p_0 p_1 & \cdots & -p_0 p_C \\
-p_1 p_0 & p_1 (1 - p_1) & \cdots & -p_1 p_C \\
\vdots & \vdots & \ddots & \vdots \\
-p_C p_0 & -p_C p_1 & \cdots & p_C (1 - p_C) \\
\end{pmatrix}$$

$$ \diamond \diamond \diamond $$

$$ L = -\sum_{c \in \mathcal{C}} y_c \log p_c $$

$$ \frac{\partial L}{\partial P} = 

- \sum_{c \in \mathcal{C}} y_c \frac{1}{p_c} = 

- [0_0, \dots, 1_n, \dots, 0_C] \cdot \begin{pmatrix}
\frac{1}{p_0} & 0 & \cdots & 0 \\
0 & \frac{1}{p_1} & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & \frac{1}{p_C} \\
\end{pmatrix} = 

-y_{onehot} \cdot \frac{1}{P}
$$

$$
\frac{\partial L}{\partial Z} = \frac{\partial L}{\partial P} \cdot \frac{\partial P}{\partial Z} = 

-y_{onehot}
\cdot 
\begin{pmatrix}
\frac{1}{p_0} & 0 & \cdots & 0 \\
0 & \frac{1}{p_1} & \cdots & 0 \\
\vdots & \vdots & \ddots & \vdots \\
0 & 0 & \cdots & \frac{1}{p_C} \\
\end{pmatrix}
\cdot
\begin{pmatrix}
p_0 (1 - p_0) & -p_0 p_1 & \cdots & -p_0 p_C \\
-p_1 p_0 & p_1 (1 - p_1) & \cdots & -p_1 p_C \\
\vdots & \vdots & \ddots & \vdots \\
-p_C p_0 & -p_C p_1 & \cdots & p_C (1 - p_C) \\
\end{pmatrix} = 

P - y_{onehot}
$$

$$ \frac{\partial L}{\partial W} = \frac{\partial L}{\partial Z} \cdot \frac{\partial Z}{\partial W} = (P - y_{onehot}) \cdot X $$

$$ \frac{\partial L}{\partial b} = \frac{\partial L}{\partial Z} \cdot \frac{\partial Z}{\partial b} = (P - y_{onehot}) \cdot 1 $$

$$ \diamond \diamond \diamond $$


In [ ]:
from torch import float32, randn, Tensor
from torch.nn import Linear
from torch.nn.functional import softmax, cross_entropy, one_hot
from torch.optim import SGD


import import_ipynb
from common import assert_eq, assert_ge, check_eq, Patient, T # type: ignore


def logistic_regression_nc_sgd_gradient(X: Tensor, Y: Tensor, epochs=2000, lr=0.1) -> tuple[float, callable]:
    """
    Perform logistic regression using Stochastic Gradient Descent (SGD) with manual gradient calculation.

    Args:
        X: Input features of shape (Samples, Features)
        Y: Target values of shape (Samples, 1)
        epochs: Number of training epochs
        lr: Learning rate

    Returns:
        A tuple containing the final loss and a prediction function that takes new input data and returns predicted probabilities.
    """

    ((s, f), c) = (X.shape, len(set(Y.squeeze().tolist())))

    assert_eq(X.shape, (s, f))
    assert_eq(Y.shape, (s, 1))

    Y = one_hot(Y.squeeze().long(), num_classes=c).float()
    assert_eq(Y.shape, (s, c))

    W = randn(f, c) * 0.1
    assert_eq(W.shape, (f, c))
    
    b = randn(c) * 0.1
    assert_eq(b.shape, (c,))

    for _ in range(epochs):
        Z = X @ W + b
        assert_eq(Z.shape, (s, c))
        
        P = softmax(Z, dim=1)
        assert_eq(P.shape, (s, c))

        dL_dZ = (P - Y) / s
        assert_eq(dL_dZ.shape, (s, c))

        dL_dW = X.T @ dL_dZ
        assert_eq(dL_dW.shape, (f, c))
        
        dL_db = dL_dZ.sum(dim=0)
        assert_eq(dL_db.shape, (c,))

        W -= lr * dL_dW
        b -= lr * dL_db

        #
        # In the autograd version, computing the loss is required because it serves 
        # as the root of the computational graph for backpropagation. 
        #
        # In the manual‑gradient version, the loss value is not needed for the weight update itself,
        # it is computed only to monitor training progress.
        #
        L = -(Y * (P + 1e-9).log()).sum(dim=1).mean()
    
    return (L.item(), lambda X: softmax(X @ W + b, dim=0))


def _test_logistic_regression_nc_sgd_gradient(epochs=2000, lr=0.1) -> None:
    """
    Test the logistic regression implementation on a synthetic dataset of patients. 
    The dataset consists of three classes: healthy (0), viral (1), and bacterial (2). 
    The model is trained on 90 samples and then evaluated on 30 samples (10 per class). 
    The test checks if the model achieves at least 90% accuracy on the evaluation set.
    """

    training_data = T([Patient([1/3, 1/3, 1/3]).data for _ in range(90)])

    X = training_data[:, :-1]
    X[:, 0] /= 100 # Temperature scaling
    X[:, 5] /= 200 # CRP scaling   

    Y = training_data[:, -1].unsqueeze(1)

    (_, model) = logistic_regression_nc_sgd_gradient(X, Y, epochs, lr)

    total = 0
    correct = 0

    for d in ([Patient([1.0, 0.0, 0.0]).data for _ in range(10)] + 
              [Patient([0.0, 1.0, 0.0]).data for _ in range(10)] + 
              [Patient([0.0, 0.0, 1.0]).data for _ in range(10)]):
        X = T(d[:-1])
        Y = T(d[-1])

        X[0] /= 100
        X[5] /= 200
        
        total += 1
        correct += check_eq(model(X).argmax(), Y)

    assert_ge(correct / total, 0.9)


if __name__ == "__main__":
    _test_logistic_regression_nc_sgd_gradient()